# Angus PRS plotting notebook

This notebook refreshes phecode labels and regenerates the Angus PRS plots from the newest saved results bundle.

It also rebuilds the association plots from the underlying score + phenotype data so you can create **ALL** and **EUR-only** versions, including regression-line plots for each midpoint phenotype.


In [ ]:
options(width = 120)

results_dir <- ""
results_root <- "results"
results_prefix <- "angus_midpoint_prs_analysis"
phecode_map_csv <- "analysis_inputs/ICD_to_Phecode_mapping.csv"
display_plots <- TRUE

score_pattern <- "METAL_midp_all_pst_eff_a1_b0.5_phi1e-02_ALL"
score_file <- ""
nightly_parquet <- "processed_data/ready_for_analysis.parquet"
covariates_parquet <- "processed_data/fitbit_cohort_covariates.parquet"
ancestry_tsv <- file.path("processed_data", "PGRS", "shared", "ancestry_preds.tsv")
pc_count <- 10L

notebook_plot_width <- 15
notebook_plot_height <- 8
options(repr.plot.width = notebook_plot_width, repr.plot.height = notebook_plot_height)


In [ ]:
required_packages <- c("arrow", "dplyr", "ggplot2", "jsonlite", "purrr", "readr", "stringr", "tibble", "tidyr")
missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_packages) > 0) {
  stop(
    paste0(
      "Missing required packages: ",
      paste(missing_packages, collapse = ", "),
      ". Install them before running the plotting notebook."
    )
  )
}

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(ggplot2)
  library(jsonlite)
  library(purrr)
  library(readr)
  library(stringr)
  library(tibble)
  library(tidyr)
})

source("prs_midpoint_collab_analysis.R", local = TRUE)

normalize_phecode <- function(x) {
  out <- as.character(x)
  out <- str_trim(out)
  out <- str_replace(out, "(\\.\\d*?[1-9])0+$", "\\1")
  out <- str_replace(out, "\\.0+$", "")
  out
}

find_latest_results_dir <- function(results_root, results_prefix) {
  if (!dir.exists(results_root)) stop("Results root not found: ", results_root)
  all_dirs <- list.dirs(results_root, recursive = FALSE, full.names = TRUE)
  candidates <- all_dirs[
    basename(all_dirs) == results_prefix |
      startsWith(basename(all_dirs), paste0(results_prefix, "_"))
  ]
  candidates <- candidates[file.exists(file.path(candidates, "tables", "phewas_results.csv"))]
  if (length(candidates) == 0) {
    stop("No Angus result directories found under ", results_root, " matching ", results_prefix)
  }
  summaries <- file.path(candidates, "summary.md")
  mtimes <- file.info(ifelse(file.exists(summaries), summaries, candidates))$mtime
  candidates[[order(mtimes, decreasing = TRUE)[[1]]]]
}

resolve_score_file <- function(score_file = "", score_pattern = "METAL_midp_all_pst_eff_a1_b0.5_phi1e-02_ALL") {
  if (nzchar(score_file)) return(score_file)
  file.path("processed_data", "PGRS", score_pattern, paste0(score_pattern, "_PGRS.txt"))
}

refresh_phewas_labels <- function(results_df, phecode_map_csv) {
  if (!file.exists(phecode_map_csv)) stop("Phecode map not found: ", phecode_map_csv)

  phemap <- readr::read_csv(phecode_map_csv, show_col_types = FALSE) %>%
    transmute(
      phecode_join = normalize_phecode(PHECODE),
      mapped_label = PHENOTYPE
    ) %>%
    distinct(phecode_join, .keep_all = TRUE)

  results_df %>%
    mutate(
      phecode = as.character(phecode),
      phecode_join = normalize_phecode(phecode)
    ) %>%
    left_join(phemap, by = "phecode_join") %>%
    mutate(
      concept_name = if_else(
        is.na(concept_name) | !nzchar(concept_name) | str_detect(concept_name, "^Phecode\\s+"),
        coalesce(mapped_label, concept_name),
        concept_name
      ),
      concept_name = if_else(is.na(concept_name) | !nzchar(concept_name), paste("Phecode", phecode), concept_name),
      label = if_else(p_value < 0.001, str_replace(concept_name, ",.*$", ""), NA_character_),
      siglevel = case_when(
        p_value < 1e-5 ~ "p < .00001",
        p_value < 1e-4 ~ "p < .0001",
        p_value < 1e-3 ~ "p < .001",
        TRUE ~ NA_character_
      ),
      siglevel = factor(siglevel, levels = c("p < .00001", "p < .0001", "p < .001"))
    ) %>%
    select(-phecode_join, -mapped_label)
}

build_association_plot_data <- function(nightly_parquet, covariates_parquet, score_file, ancestry_tsv, pc_count = 10L) {
  nightly <- arrow::read_parquet(nightly_parquet)
  phenotypes <- build_midpoint_phenotypes(nightly)
  covariates <- load_covariates(resolve_covariates_path(covariates_parquet))
  scores <- load_score_file(score_file)
  ancestry <- load_ancestry_features(ancestry_tsv, pc_count)
  analysis_df <- prepare_analysis_base(phenotypes, covariates, scores, ancestry)
  pc_cols <- intersect(names(analysis_df), paste0("pca_", seq_len(pc_count)))

  cohort_specs <- list(
    All = function(df) df,
    EUR = function(df) df %>% filter(str_to_lower(as.character(ancestry_pred)) == "eur")
  )

  assoc_results <- list()
  tertile_rows <- list()
  regression_rows <- list()
  outcome_cols <- c("person_weekend_avg_midpoint", "MSF", "MSFsc")

  for (cohort_name in names(cohort_specs)) {
    cohort_df <- cohort_specs[[cohort_name]](analysis_df)

    for (outcome in outcome_cols) {
      required <- c("score_raw", outcome, "age", "sex_concept", pc_cols)
      available_required <- intersect(required, names(cohort_df))
      df <- cohort_df %>% filter(complete.cases(across(all_of(available_required))))
      if (nrow(df) < 30) next

      df <- df %>%
        mutate(
          score_z = as.numeric(scale(score_raw)),
          score_tertile = make_score_tertiles(score_raw)
        )

      model_specs <- list(
        "PRS only" = as.formula(paste(outcome, "~ score_z")),
        "PRS + age + sex" = as.formula(paste(outcome, "~ score_z + age + sex_concept"))
      )
      if (length(pc_cols) > 0) {
        model_specs[["PRS + age + sex + PC1-PC10"]] <- as.formula(
          paste(outcome, "~ score_z + age + sex_concept +", paste(pc_cols, collapse = " + "))
        )
      }

      for (model_name in names(model_specs)) {
        fit <- stats::lm(model_specs[[model_name]], data = df)
        assoc_results[[length(assoc_results) + 1]] <- extract_linear_term(
          fit,
          term = "score_z",
          outcome = outcome,
          model_name = model_name,
          n_obs = nrow(df)
        ) %>% mutate(cohort = cohort_name)
      }

      tertile_rows[[length(tertile_rows) + 1]] <- df %>%
        transmute(
          cohort = cohort_name,
          phenotype = outcome,
          score_tertile = score_tertile,
          midpoint_hours = .data[[outcome]]
        )

      regression_rows[[length(regression_rows) + 1]] <- df %>%
        transmute(
          cohort = cohort_name,
          phenotype = outcome,
          score_raw = score_raw,
          score_z = score_z,
          midpoint_hours = .data[[outcome]]
        )
    }
  }

  forest_df <- bind_rows(assoc_results) %>%
    mutate(
      phenotype_label = factor(
        vapply(phenotype, phenotype_label, character(1)),
        levels = phenotype_label(c("person_weekend_avg_midpoint", "MSF", "MSFsc"))
      ),
      cohort = factor(cohort, levels = c("All", "EUR")),
      model = factor(model, levels = c("PRS only", "PRS + age + sex", "PRS + age + sex + PC1-PC10")),
      ci_low_minutes = ci_low_hours * 60,
      ci_high_minutes = ci_high_hours * 60
    )

  tertile_df <- bind_rows(tertile_rows) %>%
    mutate(
      phenotype_label = factor(
        vapply(phenotype, phenotype_label, character(1)),
        levels = phenotype_label(c("person_weekend_avg_midpoint", "MSF", "MSFsc"))
      ),
      cohort = factor(cohort, levels = c("All", "EUR")),
      midpoint_clock = format_clock(midpoint_hours)
    )

  regression_df <- bind_rows(regression_rows) %>%
    mutate(
      phenotype_label = factor(
        vapply(phenotype, phenotype_label, character(1)),
        levels = phenotype_label(c("person_weekend_avg_midpoint", "MSF", "MSFsc"))
      ),
      cohort = factor(cohort, levels = c("All", "EUR"))
    )

  list(forest = forest_df, tertile = tertile_df, regression = regression_df)
}


In [ ]:
chosen_results_dir <- if (nzchar(results_dir)) results_dir else find_latest_results_dir(results_root, results_prefix)
tables_dir <- file.path(chosen_results_dir, "tables")
plots_dir <- file.path(chosen_results_dir, "plots")
dir.create(plots_dir, recursive = TRUE, showWarnings = FALSE)

resolved_score_file <- resolve_score_file(score_file, score_pattern)
if (!file.exists(resolved_score_file)) stop("Score file not found: ", resolved_score_file)

required_files <- c(
  file.path(tables_dir, "cohort_flow_counts.csv"),
  file.path(tables_dir, "phewas_results.csv"),
  nightly_parquet,
  ancestry_tsv,
  resolved_score_file
)
missing <- required_files[!file.exists(required_files)]
if (length(missing) > 0) {
  stop("Missing required inputs: ", paste(missing, collapse = ", "))
}

flow_df <- readr::read_csv(file.path(tables_dir, "cohort_flow_counts.csv"), show_col_types = FALSE)
phewas_df <- readr::read_csv(file.path(tables_dir, "phewas_results.csv"), show_col_types = FALSE)
phewas_df <- refresh_phewas_labels(phewas_df, phecode_map_csv) %>%
  arrange(p_value) %>%
  mutate(
    minus_log10_p = -log10(p_value),
    phecode_index = row_number()
  )

association_plot_data <- build_association_plot_data(
  nightly_parquet = nightly_parquet,
  covariates_parquet = covariates_parquet,
  score_file = resolved_score_file,
  ancestry_tsv = ancestry_tsv,
  pc_count = pc_count
)

readr::write_csv(phewas_df, file.path(tables_dir, "phewas_results.csv"))
readr::write_csv(phewas_df, file.path(tables_dir, "phewas_manhattan_plot_data.csv"))
readr::write_csv(phewas_df %>% filter(is.finite(odds_ratio)), file.path(tables_dir, "phewas_volcano_plot_data.csv"))
readr::write_csv(association_plot_data$forest, file.path(tables_dir, "association_forest_plot_data_by_cohort.csv"))
readr::write_csv(association_plot_data$tertile, file.path(tables_dir, "midpoint_by_prs_tertile_plot_data_by_cohort.csv"))
readr::write_csv(association_plot_data$regression, file.path(tables_dir, "midpoint_regression_plot_data_by_cohort.csv"))

cat("Using results directory:", chosen_results_dir, "\n")
cat("Using score file:", resolved_score_file, "\n")


In [ ]:
# Cohort flow figure
edge_df <- flow_df %>%
  filter(!is.na(parent_step_id)) %>%
  left_join(
    flow_df %>% select(step_id, x_parent = x, y_parent = y),
    by = c("parent_step_id" = "step_id")
  )

p_flow <- ggplot() +
  geom_segment(
    data = edge_df,
    aes(x = x_parent + 0.18, y = y_parent, xend = x - 0.18, yend = y, color = branch),
    arrow = grid::arrow(length = grid::unit(0.18, "cm")),
    linewidth = 0.7,
    lineend = "round"
  ) +
  geom_label(
    data = flow_df,
    aes(x = x, y = y, label = label, fill = branch),
    linewidth = 0.25,
    size = 3.8,
    label.padding = grid::unit(0.18, "lines")
  ) +
  scale_fill_manual(values = c("Sleep" = "#d9edf7", "Association" = "#d5e8d4", "PheWAS" = "#fce5cd")) +
  scale_color_manual(values = c("Sleep" = "#2c7fb8", "Association" = "#238b45", "PheWAS" = "#d95f0e")) +
  coord_cartesian(xlim = c(0.7, 4.35), ylim = c(0, 3.6), clip = "off") +
  labs(
    title = "Participant flow for Angus midpoint PRS analysis",
    subtitle = "Largest available sleep-genetics cohort is used for association; PheWAS uses the sleep-genetics-EHR overlap",
    caption = "Source table: tables/cohort_flow_counts.csv"
  ) +
  theme_void() +
  theme(
    plot.title = element_text(face = "bold", size = 16),
    plot.subtitle = element_text(size = 12, color = "gray30"),
    plot.caption = element_text(size = 9, color = "gray35"),
    legend.position = "none",
    plot.margin = margin(15, 25, 15, 25)
  )

flow_path <- file.path(plots_dir, "cohort_flow.png")
ggsave(flow_path, plot = p_flow, width = 14, height = 7, dpi = 320, bg = "white")
if (display_plots) p_flow else flow_path


In [ ]:
# Association forest plot (ALL)
forest_all <- association_plot_data$forest %>% filter(cohort == "All")
p_forest_all <- forest_all %>%
  ggplot(aes(x = estimate_minutes, y = phenotype_label, color = model)) +
  geom_point(position = position_dodge(width = 0.5), size = 2.3) +
  geom_errorbar(
    aes(xmin = ci_low_minutes, xmax = ci_high_minutes),
    position = position_dodge(width = 0.5),
    width = 0.2,
    orientation = "y"
  ) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray40") +
  labs(
    title = "Association of PRS per SD with midpoint phenotypes",
    subtitle = "All ancestries",
    x = "Beta (minutes per SD higher PRS)",
    y = NULL,
    color = "Model",
    caption = "Source table: tables/association_forest_plot_data_by_cohort.csv"
  ) +
  theme_research()

forest_all_path <- file.path(plots_dir, "association_forest_per_sd.png")
ggsave(forest_all_path, plot = p_forest_all, width = 12, height = 6, dpi = 320, bg = "white")
if (display_plots) p_forest_all else forest_all_path


In [ ]:
# Association forest plot (EUR)
forest_eur <- association_plot_data$forest %>% filter(cohort == "EUR")
p_forest_eur <- forest_eur %>%
  ggplot(aes(x = estimate_minutes, y = phenotype_label, color = model)) +
  geom_point(position = position_dodge(width = 0.5), size = 2.3) +
  geom_errorbar(
    aes(xmin = ci_low_minutes, xmax = ci_high_minutes),
    position = position_dodge(width = 0.5),
    width = 0.2,
    orientation = "y"
  ) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray40") +
  labs(
    title = "Association of PRS per SD with midpoint phenotypes",
    subtitle = "EUR only",
    x = "Beta (minutes per SD higher PRS)",
    y = NULL,
    color = "Model",
    caption = "Source table: tables/association_forest_plot_data_by_cohort.csv"
  ) +
  theme_research()

forest_eur_path <- file.path(plots_dir, "association_forest_per_sd_eur.png")
ggsave(forest_eur_path, plot = p_forest_eur, width = 12, height = 6, dpi = 320, bg = "white")
if (display_plots) p_forest_eur else forest_eur_path


In [ ]:
# Tertile plot (ALL)
tertile_all <- association_plot_data$tertile %>% filter(cohort == "All")
p_tertile_all <- ggplot(tertile_all, aes(x = score_tertile, y = midpoint_hours, fill = score_tertile)) +
  geom_boxplot(outlier.alpha = 0.2, width = 0.68, color = "gray25") +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 2.7, fill = "gold", color = "black") +
  facet_wrap(~ phenotype_label, scales = "free_y") +
  scale_fill_manual(values = c("Low" = "#c6dbef", "Medium" = "#6baed6", "High" = "#2171b5")) +
  labs(
    title = "Midpoint phenotypes by PRS tertile",
    subtitle = "All ancestries",
    x = "PRS tertile",
    y = "Midpoint (linearized decimal hours)",
    caption = "Source table: tables/midpoint_by_prs_tertile_plot_data_by_cohort.csv"
  ) +
  theme_research() +
  theme(legend.position = "none")

tertile_all_path <- file.path(plots_dir, "midpoint_by_prs_tertile.png")
ggsave(tertile_all_path, plot = p_tertile_all, width = 13, height = 7, dpi = 320, bg = "white")
if (display_plots) p_tertile_all else tertile_all_path


In [ ]:
# Tertile plot (EUR)
tertile_eur <- association_plot_data$tertile %>% filter(cohort == "EUR")
p_tertile_eur <- ggplot(tertile_eur, aes(x = score_tertile, y = midpoint_hours, fill = score_tertile)) +
  geom_boxplot(outlier.alpha = 0.2, width = 0.68, color = "gray25") +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 2.7, fill = "gold", color = "black") +
  facet_wrap(~ phenotype_label, scales = "free_y") +
  scale_fill_manual(values = c("Low" = "#c6dbef", "Medium" = "#6baed6", "High" = "#2171b5")) +
  labs(
    title = "Midpoint phenotypes by PRS tertile",
    subtitle = "EUR only",
    x = "PRS tertile",
    y = "Midpoint (linearized decimal hours)",
    caption = "Source table: tables/midpoint_by_prs_tertile_plot_data_by_cohort.csv"
  ) +
  theme_research() +
  theme(legend.position = "none")

tertile_eur_path <- file.path(plots_dir, "midpoint_by_prs_tertile_eur.png")
ggsave(tertile_eur_path, plot = p_tertile_eur, width = 13, height = 7, dpi = 320, bg = "white")
if (display_plots) p_tertile_eur else tertile_eur_path


In [ ]:
# Regression line plot (ALL)
reg_all <- association_plot_data$regression %>% filter(cohort == "All")
p_reg_all <- ggplot(reg_all, aes(x = score_z, y = midpoint_hours)) +
  geom_point(alpha = 0.08, size = 0.55, color = "#2c7fb8") +
  geom_smooth(method = "lm", se = TRUE, color = "#cb181d", linewidth = 0.9) +
  facet_wrap(~ phenotype_label, scales = "free_y") +
  labs(
    title = "PRS-midpoint regression lines",
    subtitle = "All ancestries; unadjusted linear fit for each midpoint phenotype",
    x = "PRS score (z-scored)",
    y = "Midpoint (linearized decimal hours)",
    caption = "Source table: tables/midpoint_regression_plot_data_by_cohort.csv"
  ) +
  theme_research()

reg_all_path <- file.path(plots_dir, "midpoint_regression_lines.png")
ggsave(reg_all_path, plot = p_reg_all, width = 13, height = 7, dpi = 320, bg = "white")
if (display_plots) p_reg_all else reg_all_path


In [ ]:
# Regression line plot (EUR)
reg_eur <- association_plot_data$regression %>% filter(cohort == "EUR")
p_reg_eur <- ggplot(reg_eur, aes(x = score_z, y = midpoint_hours)) +
  geom_point(alpha = 0.08, size = 0.55, color = "#2c7fb8") +
  geom_smooth(method = "lm", se = TRUE, color = "#cb181d", linewidth = 0.9) +
  facet_wrap(~ phenotype_label, scales = "free_y") +
  labs(
    title = "PRS-midpoint regression lines",
    subtitle = "EUR only; unadjusted linear fit for each midpoint phenotype",
    x = "PRS score (z-scored)",
    y = "Midpoint (linearized decimal hours)",
    caption = "Source table: tables/midpoint_regression_plot_data_by_cohort.csv"
  ) +
  theme_research()

reg_eur_path <- file.path(plots_dir, "midpoint_regression_lines_eur.png")
ggsave(reg_eur_path, plot = p_reg_eur, width = 13, height = 7, dpi = 320, bg = "white")
if (display_plots) p_reg_eur else reg_eur_path


In [ ]:
# PheWAS Manhattan plot
p_manhattan <- ggplot(phewas_df, aes(x = phecode_index, y = minus_log10_p, color = fdr < 0.05)) +
  geom_point(alpha = 0.85, size = 1.9) +
  geom_hline(
    yintercept = -log10(0.05 / max(nrow(phewas_df), 1)),
    linetype = "dashed",
    color = "firebrick"
  ) +
  scale_color_manual(values = c("FALSE" = "gray55", "TRUE" = "#1b9e77")) +
  labs(
    title = "Continuous PRS per SD PheWAS",
    subtitle = "Green points pass FDR < 0.05; dashed line shows Bonferroni threshold",
    x = "Phecode index",
    y = expression(-log[10](p)),
    color = "FDR < 0.05",
    caption = "Source table: tables/phewas_manhattan_plot_data.csv"
  ) +
  theme_research()

manhattan_path <- file.path(plots_dir, "phewas_manhattan.png")
ggsave(manhattan_path, plot = p_manhattan, width = 13, height = 7, dpi = 320, bg = "white")
if (display_plots) p_manhattan else manhattan_path


In [ ]:
# PheWAS volcano plot
volcano_df <- phewas_df %>% filter(is.finite(odds_ratio))
p_volcano <- ggplot(volcano_df, aes(x = odds_ratio, y = minus_log10_p)) +
  geom_point(color = "grey70", size = 1.9, alpha = 0.85) +
  geom_point(
    data = volcano_df %>% filter(!is.na(siglevel)),
    aes(color = siglevel),
    size = 2.2,
    alpha = 0.95
  ) +
  geom_vline(xintercept = 1, color = "gray35", linewidth = 0.6) +
  geom_hline(yintercept = -log10(0.05), color = "#b2182b", linetype = "dashed", linewidth = 0.6) +
  geom_hline(
    yintercept = -log10(0.05 / max(nrow(phewas_df), 1)),
    color = "#2166ac",
    linetype = "dashed",
    linewidth = 0.6
  ) +
  geom_text(
    data = volcano_df %>% filter(!is.na(label)),
    aes(label = label),
    check_overlap = TRUE,
    nudge_y = 0.1,
    size = 3.2
  ) +
  scale_color_manual(
    values = c("p < .00001" = "#7f0000", "p < .0001" = "#cb181d", "p < .001" = "#fb6a4a"),
    na.translate = FALSE,
    drop = FALSE
  ) +
  labs(
    title = "PheWAS volcano plot for continuous PRS",
    subtitle = "Legacy-style odds-ratio display; labels shown for p < 0.001; all finite ORs plotted",
    x = "Odds ratio per 1 SD higher PRS",
    y = expression(-log[10](p)),
    color = "Sig. level",
    caption = "Source table: tables/phewas_volcano_plot_data.csv"
  ) +
  theme_research()

volcano_path <- file.path(plots_dir, "phewas_volcano.png")
ggsave(volcano_path, plot = p_volcano, width = 13, height = 8, dpi = 320, bg = "white")
if (display_plots) p_volcano else volcano_path


In [ ]:
top_rows <- phewas_df %>%
  select(phecode, concept_name, odds_ratio, p_value, fdr) %>%
  slice_head(n = 15)
top_rows
